# NammaSpace — Dense Reconstruction on **Kaggle** (COLMAP + 3DGS)

Same pipeline as the Colab notebook, but on Kaggle's free GPU (T4×2 / P100, 30 hrs/week,
independent of Colab's quota). Uses **Nerfstudio `splatfacto`** (prebuilt wheels) so there is
**no CUDA rasterizer to compile**. Output is a `.ply` you download and compress to `.ksplat`.

## Before you run (do these once)
1. **Settings (right panel) → Accelerator → `GPU T4 x2`** (or P100).
2. **Settings → Internet → `On`** (needs a phone-verified Kaggle account — required for pip/apt).
3. **Add your video as data:** right panel → **Add Data → Datasets → Upload** → drop your `.mp4`.
   It mounts read-only under `/kaggle/input/<your-dataset>/`.
   > Upload a **trimmed / 1600px** copy (~100–200 MB), not the raw 1.5 GB file — far faster, and
   > 1600px is all the reconstruction uses.

Then run cells top to bottom.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Find your video (auto-detects the .mp4 you added as a Dataset)
Hard-fails if the file is missing or near-empty — so a broken upload can't silently waste a run.

In [ ]:
import glob, os
SCENE_ID = 'apartment'            # <-- name for this scene (e.g. 'room2' or 'apartment')
WORK = '/kaggle/working/ns'
os.makedirs(WORK, exist_ok=True)

cands = sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True) +
               glob.glob('/kaggle/input/**/*.MP4', recursive=True))
assert cands, 'No .mp4 under /kaggle/input — use "Add Data → Datasets → Upload" to add your video.'
VIDEO = cands[0]
sz = os.path.getsize(VIDEO) / 1e6
assert sz > 1, f'Video is only {sz:.2f} MB — the upload looks empty/broken. Re-upload the dataset.'
print(f'using {VIDEO}   {sz:.1f} MB')

## 3. Install COLMAP + Nerfstudio
~8–12 min the first time. (ffmpeg is already on the Kaggle image.) Nerfstudio bundles frame
extraction, COLMAP SfM, `splatfacto` training and splat export.

In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap > /dev/null && echo 'colmap ok'
!pip -q install nerfstudio 2>&1 | tail -3
print('\nchecks:'); !which ffmpeg colmap ns-process-data ns-train

## 4. Video → frames → camera poses (COLMAP)
`--num-frames-target` ~300 keeps overlap without overloading SfM. Raise it for a whole apartment.

In [ ]:
!ns-process-data video --data "$VIDEO" \
    --output-dir "$WORK/$SCENE_ID" \
    --num-frames-target 300 \
    --sfm-tool colmap --matching-method sequential

## 5. Train the dense Gaussian splat (splatfacto)
30k steps. `cull-alpha-thresh` prunes low-opacity **floaters** (raise toward 0.1–0.15 if hazy).

In [ ]:
!ns-train splatfacto \
    --data "$WORK/$SCENE_ID" \
    --output-dir "$WORK/outputs" \
    --max-num-iterations 30000 \
    --pipeline.model.cull-alpha-thresh 0.1 \
    --viewer.quit-on-train-completion True

## 6. Export the splat as .ply

In [ ]:
import glob
cfg = sorted(glob.glob(f'{WORK}/outputs/**/config.yml', recursive=True))[-1]
print('using config:', cfg)
!ns-export gaussian-splat --load-config "$cfg" --output-dir "$WORK/export"
!ls -lh $WORK/export

## 7. Save the result where you can download it
`/kaggle/working/` is the only writable, persisted folder. After the run, grab the files from the
**Output** tab (right panel), or click **Save Version** to snapshot them.

In [ ]:
out = f'/kaggle/working/out/{SCENE_ID}'
os.makedirs(out, exist_ok=True)
!cp $WORK/export/*.ply "$out/"
# also save the COLMAP poses (needed for cameras.json / metric scale in the scene contract)
!cp -r $WORK/$SCENE_ID/colmap/sparse "$out/" 2>/dev/null; cp -r $WORK/$SCENE_ID/sparse "$out/" 2>/dev/null
print('saved to', out); !ls -lhR "$out"

## 8. Back on your laptop
Download `out/<scene>/*.ply` from the **Output** tab, then compress + deploy:
```bash
node viewer/convert-ksplat.mjs <downloaded>.ply viewer/public/scenes/<scene>/splat/scene.ksplat
```
Then build `scene.json` (and `cameras.json` from the saved COLMAP `sparse/`) per
`docs/round2-scene-contract.md`, and view with `?scene=<scene>`.

**Tips:** raise `cull-alpha-thresh` (step 5) if hazy; more frames → denser multi-room coverage;
capture per the SOP (sharp frames + parallax).